<a href="https://colab.research.google.com/github/Tuchobm/Curso-IA-Google-Colab/blob/main/6_4_herramientas_example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ejercicio 4: Agente de IA (Herramientas)

En este ejercicio, crearemos un agente de IA que pueda interactuar con el usuario y responder a preguntas usando una serie de herramientas. El objetivo es que el agente pueda realizar tareas específicas utilizando estas herramientas y responder a preguntas de manera efectiva.

Además, encontrarás partes del código contendrán el comentario de '# ACTIVIDAD' que indican dónde debes completar el código o realizar tareas específicas. Asegúrate de seguir las instrucciones y completar el código donde se indique.

# Cargar el modelo y el tokenizer

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch


model_name = "NousResearch/Hermes-2-Pro-Llama-3-8B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/56.1k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/700 [00:00<?, ?B/s]

...

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]

# Herramientas

In [ ]:
# ACTIVIDAD: Crea nuevas herramientas para el modelo.
# Teneis que documentar las herramientas que creeis necesarias para el modelo.
# Basaros en formato que observais en la funciones ejemplo.
# Libertad total para crear nuevas herramientas.
import requests


def get_current_temperature(location: str):
    """Devuelve la temperatura actual en grados Celsius.

    Args:
      location: Ubicación a revisar temperatura (Predeterminado Madrid)
    """
    # Coordenadas de Madrid: 40.4168° N, 3.7038° W

    api_key = "09f8cdcb16f7a023e71df5b6cf6caa8d"

    request = requests.get(
        "http://api.openweathermap.org/geo/1.0/direct",
        params={
            "q": location,
            "limit": 1,
            "appid": api_key,
        },
    )
    coordinates = request.json()

    if request.status_code != 200:
        print("Error al obtener las coordenadas, motivo:", coordinates)
        coordinates = [{"lat": 40.4168, "lon": -3.7038}]

    lattitude = coordinates[0]["lat"]
    longitude = coordinates[0]["lon"]

    request = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": lattitude,
            "longitude": longitude,
            "current": "temperature",
        },
    )

    if request.status_code != 200:
        raise ValueError("Error al obtener la temperatura")
    return request.json()["current"]["temperature"]


def add(a: str, b: str) -> float:
    """Suma dos números.

    Args:
        a: Primer número.
        b: Segundo número.
    """
    return float(a) + float(b)


tools = [get_current_temperature, add]

## Conversación con el modelo

In [ ]:
import json

system_prompt = """
Eres un assistente inteligente que puede realizar tareas y responder preguntas.
"""
temperature = 0.8
top_p = 0.95
max_new_tokens = 256

messages = [{"role": "system", "content": system_prompt}]
while True:
    if len(messages) == 0 or messages[-1]["role"] != "tool":
        user_input = input("Usuario (Escribe 'exit' para salir): ")
        if user_input.lower() == "exit" or user_input.lower() == "":
            break
        messages.append({"role": "user", "content": user_input})
    prompt = tokenizer.apply_chat_template(
        messages,
        tools=tools,
        return_tensors="pt",
        return_dict=True,
        add_generation_prompt=True,
    )
    prompt = prompt.to(model.device)

    out = model.generate(
        **prompt,
        temperature=temperature,
        top_p=top_p,
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.eos_token_id,
    )
    generated_text_ids = out[0, prompt["input_ids"].shape[1] :]
    generated_text = tokenizer.decode(generated_text_ids)
    response = tokenizer.decode(generated_text_ids, skip_special_tokens=True)

    messages.append({"role": "assistant", "content": response})

    if "<tool_call>" not in generated_text:
        print("Asistente: ", response)
    else:
        # Si se detecta una llamada a la herramienta, extraer el nombre de la herramienta y los argumentos
        tool_call = generated_text.split("<tool_call>")[1].split("</tool_call>")[0]
        tool_call = json.loads(tool_call)

        tool_name = tool_call["name"]
        tool = next((t for t in tools if t.__name__ == tool_name), None)
        if tool:
            # Llamar a la herramienta con los argumentos proporcionados
            print(f"Llamando a la herramienta '{tool_name}' con argumentos: {tool_call['arguments']}")
            try:
              result = tool(**tool_call["arguments"])
            except Exception as e:
              result = f"Error: `{e}`"
            print(f"Resultado de la herramienta '{tool_name}': {result}")
            messages.append({"role": "tool", "name": tool_name, "content": str(result)})
        else:
            print(f"Herramienta '{tool_name}' no encontrada.")
            messages.append(
                {
                    "role": "tool",
                    "name": tool_name,
                    "content": "Herramienta no encontrada.",
                }
            )

Usuario (Escribe 'exit' para salir): tiempo en barcelona
Llamando a la herramienta 'get_current_temperature' con argumentos: {'location': 'Barcelona'}
Resultado de la herramienta 'get_current_temperature': 14.9
Asistente:  La temperatura actual en Barcelona es de 14.9°C.
Usuario (Escribe 'exit' para salir): Bilbao
Llamando a la herramienta 'get_current_temperature' con argumentos: {'location': 'Bilbao'}
Resultado de la herramienta 'get_current_temperature': 13.3
Asistente:  La temperatura actual en Bilbao es de 13.3°C.
Usuario (Escribe 'exit' para salir): exit
